# Phase 24 — frozen cross-market robustness

This notebook tests whether the structural representation and GBP-selected model specification survive fixed market, period, sampling-frequency, and threshold-ladder changes. These symbols and histories have already been explored, so the results are robustness evidence—not untouched external confirmation.

In [1]:
import os

# Reproducible HMM fits (M15): fix the BLAS/OpenMP and PyTorch thread counts before the numerical
# libraries load. A concurrent or differently threaded run can change floating-point reduction order
# and therefore the EM path. Restart the kernel after changing HMM_THREADS.
HMM_THREADS = 1
for _thread_variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_thread_variable, str(HMM_THREADS))

import gc

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from dc_basics.data import resolve_data_root
from dc_multiscale_research import (
    DEFAULT_MAX_POINT_MASS_SHARE,
    FeatureTier,
    ModelFamily,
    ObservationClock,
    add_future_diagnostics,
    build_multiscale_state,
    incremental_usefulness,
    load_close,
    permutation_omega_test,
    plot_price_states,
    prepare_data,
    ratio_ladder,
    select_hmm_candidates,
)

torch.set_num_threads(HMM_THREADS)

DATA_ROOT = resolve_data_root()
MARKETS = {
    "GBPUSD.r": "fx",
    "EURUSD.r": "fx",
    "USDJPY.r": "fx",
    "US500.r": "equity_index",
    "XAUUSD.r": "metal",
}
PILOT_SYMBOL = "GBPUSD.r"
SAMPLING_MINUTES = (5, 15, 30)
LADDER_STARTS = (0.0015, 0.0020, 0.0025)
LADDER_RATIO = 1.5
THRESHOLD_COUNT = 5
PRIMARY_THRESHOLDS = ratio_ladder(0.0020, LADDER_RATIO, THRESHOLD_COUNT)
MINIMUM_OUTER_EVENTS = 75
CLOCK = ObservationClock.UNION_EVENT
FEATURE_TIER = FeatureTier.DYNAMIC
STATE_COUNTS = (3, 4, 5)
HMM_FAMILIES = (ModelFamily.NORMAL_HMM, ModelFamily.STUDENTT_HMM)
RANDOM_STATE = 17
STABILITY_SEEDS = (29, 47)
MAX_ITER = 50
N_RESTARTS = 3
MINIMUM_VALIDATION_OCCUPANCY = 0.02
MINIMUM_RESTART_ARI = 0.80
MIN_EMISSION_VARIANCE = 1e-2
N_PERMUTATIONS = 499
N_BOOTSTRAP = 499
BOOTSTRAP_BLOCK_ROWS = 20
BOOTSTRAP_CONFIDENCE = 0.90
PLOT_RECENT_PRIMARY_CASES = True
LAST_N = 4_000

## Frozen robustness matrix

Structural sensitivity uses the full Cartesian product of five symbols, three close-sampling intervals, and three constant-ratio ladders. No case is dropped because its result looks inconvenient. Insufficient outer-event cases are reported rather than repaired with a symbol-specific threshold.

All symbols are restricted to their common available calendar interval. The native FXPIG bars are stamped at their **open** (MetaTrader convention), and resampling keeps that convention: a 15- or 30-minute bar is labelled by its interval open and carries the last native close inside the interval (`load_close(..., bar_timestamp="open")`). Early, middle, and late periods are the three equal **calendar** thirds of the common interval, so every market and sampling frequency is split at the same dates.

In [2]:
native_prices = {symbol: load_close(DATA_ROOT, symbol) for symbol in MARKETS}
common_start = max(series.index[0] for series in native_prices.values())
common_end = min(series.index[-1] for series in native_prices.values())
print(f"Common interval: {common_start} to {common_end}")
PERIOD_LABELS = ["early", "middle", "late"]
PERIOD_EDGES = pd.date_range(common_start, common_end, periods=len(PERIOD_LABELS) + 1)
print("Calendar thirds:", [f"{start:%Y-%m-%d} to {end:%Y-%m-%d}" for start, end in zip(PERIOD_EDGES[:-1], PERIOD_EDGES[1:])])

market_contract = pd.DataFrame.from_records([
    {
        "symbol": symbol,
        "market_class": market_class,
        "native_bars": len(native_prices[symbol].loc[common_start:common_end]),
    }
    for symbol, market_class in MARKETS.items()
]).set_index("symbol")
display(market_contract)

Common interval: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


,market_class,native_bars
symbol,,
GBPUSD.r,fx,258573
EURUSD.r,fx,258848
USDJPY.r,fx,258667
US500.r,equity_index,245467
XAUUSD.r,metal,244950


## Structural robustness across 45 predeclared cases

In [3]:
structural_records = []
for symbol, market_class in MARKETS.items():
    for sampling_minutes in SAMPLING_MINUTES:
        sampled = load_close(DATA_ROOT, symbol, sampling_minutes).loc[common_start:common_end]
        for ladder_start in LADDER_STARTS:
            thresholds = ratio_ladder(ladder_start, LADDER_RATIO, THRESHOLD_COUNT)
            _, global_state, _ = build_multiscale_state(
                sampled, thresholds, include_scale_state=False
            )
            period_codes = PERIOD_EDGES[1:-1].searchsorted(global_state.index, side="right")
            thirds = pd.Categorical.from_codes(period_codes, categories=PERIOD_LABELS)
            by_third = global_state.assign(sample_third=thirds).groupby("sample_third", observed=False).agg(
                broad_share=("broad_agreement", "mean"),
                fragmented_share=("coherent_frontier", lambda values: 1.0 - values.mean()),
            )
            outer_events = int(global_state[f"dcc_flag_{THRESHOLD_COUNT - 1:02d}"].sum())
            structural_records.append({
                "symbol": symbol,
                "market_class": market_class,
                "sampling_minutes": sampling_minutes,
                "ladder_start": ladder_start,
                "outer_threshold": thresholds[-1],
                "initialized_bars": len(global_state),
                "observed_configurations": global_state["configuration_code"].nunique(),
                "broad_share": global_state["broad_agreement"].mean(),
                "fragmented_share": (~global_state["coherent_frontier"]).mean(),
                "union_event_share": global_state["union_event"].mean(),
                "outer_events": outer_events,
                "adequate_outer_events": outer_events >= MINIMUM_OUTER_EVENTS,
                "broad_share_period_range": by_third["broad_share"].max() - by_third["broad_share"].min(),
                "fragmented_share_period_range": (
                    by_third["fragmented_share"].max() - by_third["fragmented_share"].min()
                ),
            })
            del global_state
            gc.collect()

structural_robustness = pd.DataFrame.from_records(structural_records)
display(structural_robustness)
print("Cases:", len(structural_robustness))
print("Adequate outer-event cases:", structural_robustness["adequate_outer_events"].sum())

,symbol,market_class,sampling_minutes,ladder_start,outer_threshold,initialized_bars,observed_configurations,broad_share,fragmented_share,union_event_share,outer_events,adequate_outer_events,broad_share_period_range,fragmented_share_period_range
0,GBPUSD.r,fx,5,0.0015,0.007594,258007,32,0.263384,0.291810,0.031724,254,True,0.043077,0.059606
1,GBPUSD.r,fx,5,0.0020,0.010125,257813,32,0.263885,0.293709,0.019634,146,True,0.051863,0.044904
2,GBPUSD.r,fx,5,0.0025,0.012656,257639,32,0.268546,0.271780,0.013391,101,True,0.039151,0.014183
3,GBPUSD.r,fx,15,0.0015,0.007594,86219,32,0.290249,0.268131,0.073858,230,True,0.043459,0.032463
4,GBPUSD.r,fx,15,0.0020,0.010125,86152,32,0.284671,0.276198,0.048252,132,True,0.041857,0.023375
5,GBPUSD.r,fx,15,0.0025,0.012656,85637,32,0.284281,0.264232,0.034179,96,True,0.060545,0.028402
6,GBPUSD.r,fx,30,0.0015,0.007594,43194,32,0.304278,0.260013,0.119230,208,True,0.049937,0.057925
7,GBPUSD.r,fx,30,0.0020,0.010125,43168,32,0.300361,0.259776,0.081565,126,True,0.049204,0.022952
8,GBPUSD.r,fx,30,0.0025,0.012656,42910,32,0.290608,0.254416,0.058751,94,True,0.061176,0.021236
9,EURUSD.r,fx,5,0.0015,0.007594,258137,32,0.248922,0.303327,0.028942,238,True,0.027137,0.023159


Cases: 45
Adequate outer-event cases: 45


## Robustness summaries without case deletion

The first table asks whether market classes occupy structurally different hierarchies under the same grid. The second shows threshold and sampling sensitivity. Adequacy failures remain visible.

In [4]:
primary_cases = structural_robustness.loc[
    structural_robustness["sampling_minutes"].eq(5)
    & structural_robustness["ladder_start"].eq(0.0020)
]
display(primary_cases.set_index("symbol").style.format({
    "ladder_start": "{:.3%}",
    "outer_threshold": "{:.3%}",
    "broad_share": "{:.2%}",
    "fragmented_share": "{:.2%}",
    "union_event_share": "{:.2%}",
    "broad_share_period_range": "{:.2%}",
    "fragmented_share_period_range": "{:.2%}",
}))

sensitivity = structural_robustness.groupby(["sampling_minutes", "ladder_start"]).agg(
    cases=("symbol", "size"),
    adequate_cases=("adequate_outer_events", "sum"),
    median_outer_events=("outer_events", "median"),
    median_broad_share=("broad_share", "median"),
    median_fragmented_share=("fragmented_share", "median"),
)
display(sensitivity.style.format({
    "median_outer_events": "{:.1f}",
    "median_broad_share": "{:.2%}",
    "median_fragmented_share": "{:.2%}",
}))

,market_class,sampling_minutes,ladder_start,outer_threshold,initialized_bars,observed_configurations,broad_share,fragmented_share,union_event_share,outer_events,adequate_outer_events,broad_share_period_range,fragmented_share_period_range
symbol,,,,,,,,,,,,,
GBPUSD.r,fx,5,0.200%,1.012%,257813,32,26.39%,29.37%,1.96%,146,True,5.19%,4.49%
EURUSD.r,fx,5,0.200%,1.012%,258030,32,24.63%,31.00%,1.79%,144,True,2.69%,1.45%
USDJPY.r,fx,5,0.200%,1.012%,257979,32,32.65%,27.79%,2.75%,252,True,9.17%,2.33%
US500.r,equity_index,5,0.200%,1.012%,245113,32,33.57%,26.13%,5.26%,537,True,1.52%,0.63%
XAUUSD.r,metal,5,0.200%,1.012%,244719,32,28.37%,28.44%,8.55%,822,True,3.90%,3.03%


## Freeze the model family/state count on GBP validation

The specification is selected once on the pilot after prefix, occupancy, and restart-stability eligibility gates. External symbols do not choose their own family, number of states, or emission features: the feature list screened on GBP training rows is passed unchanged to every market, and any frozen feature that is zero-inflated in another market is listed rather than silently dropped. Each external model may estimate emissions and transitions on its own training partition using the same algorithm and hyperparameters; this is frozen-procedure refitting, not direct parameter transfer.

In [5]:
pilot_prices = native_prices[PILOT_SYMBOL].loc[common_start:common_end]
_, pilot_global, _ = build_multiscale_state(
    pilot_prices, PRIMARY_THRESHOLDS, include_scale_state=False
)
pilot_prepared = prepare_data(pilot_global, CLOCK, FEATURE_TIER)
pilot_selection_result = select_hmm_candidates(
    pilot_prepared,
    HMM_FAMILIES,
    STATE_COUNTS,
    random_state=RANDOM_STATE,
    stability_seeds=STABILITY_SEEDS,
    max_iter=MAX_ITER,
    n_restarts=N_RESTARTS,
    minimum_validation_occupancy=MINIMUM_VALIDATION_OCCUPANCY,
    minimum_restart_ari=MINIMUM_RESTART_ARI,
    min_emission_variance=MIN_EMISSION_VARIANCE,
)
pilot_fit = pilot_selection_result.selected_fit
if pilot_fit is None:
    raise RuntimeError("No pilot HMM passed the frozen eligibility rules.")
SELECTED_FAMILY = pilot_fit.family
SELECTED_N_STATES = pilot_fit.n_states
display(pilot_selection_result.table.set_index(["family", "n_states"]))
display(pilot_selection_result.stability.set_index(["family", "n_states", "seed"]))
print(f"Frozen external procedure: {SELECTED_FAMILY.value}, K={SELECTED_N_STATES}")

clock     tier  feature_count  \
family       n_states                                             
normal_hmm   3         union_event_time  dynamic             21   
             4         union_event_time  dynamic             21   
             5         union_event_time  dynamic             21   
studentt_hmm 3         union_event_time  dynamic             21   
             4         union_event_time  dynamic             21   
             5         union_event_time  dynamic             21   

                       removed_feature_count  prepared_rows_dropped error  \
family       n_states                                                       
normal_hmm   3                             1                      0  None   
             4                             1                      0  None   
             5                             1                      0  None   
studentt_hmm 3                             1                      0  None   
             4                             1                      0  None   
             5                             1                      0  None   

                       validation_log_likelihood_per_row  \
family       n_states                                      
normal_hmm   3                                -22.590537   
             4                                -19.738444   
             5                                -18.732857   
studentt_hmm 3                                -22.896883   
             4                                -21.071885   
             5                                -20.080213   

                       minimum_state_occupancy  switch_rate  median_dwell  \
family       n_states                                                       
normal_hmm   3                        0.292111     0.947705           1.0   
             4                        0.156716     0.896478           1.0   
             5                        0.126866     0.935966           1.0   
studentt_hmm 3                        0.239872     0.788687           1.0   
             4                        0.124733     0.872999           1.0   
             5                        0.131130     0.899680           1.0   

                       ...  baseline_nmi  semantic_mapping_coverage  \
family       n_states  ...                                            
normal_hmm   3         ...      0.330369                   1.000000   
             4         ...      0.511658                   1.000000   
             5         ...      0.266190                   0.786780   
studentt_hmm 3         ...      0.201646                   1.000000   
             4         ...      0.387654                   1.000000   
             5         ...      0.439902                   0.760128   

                       baseline_mapped_agreement  minimum_fitted_variance  \
family       n_states                                                       
normal_hmm   3                          0.479744                     0.01   
             4                          0.651386                     0.01   
             5                          0.510840                     0.01   
studentt_hmm 3                          0.473348                     0.01   
             4                          0.603412                     0.01   
             5                          0.807854                     0.01   

                       variance_floor_hit_count  variance_floor_hit_share  \
family       n_states                                                       
normal_hmm   3                                3                  0.047619   
             4                                5                  0.059524   
             5                               13                  0.123810   
studentt_hmm 3                                5                  0.079365   
             4                                5                  0.059524   
             5                                6          

error  validation_ari_vs_primary  \
family       n_states seed                                    
normal_hmm   3        29    None                   0.769205   
                      47    None                   0.668122   
             4        29    None                   1.000000   
                      47    None                   1.000000   
             5        29    None                   0.545847   
                      47    None                   0.783602   
studentt_hmm 3        29    None                   1.000000   
                      47    None                   0.724244   
             4        29    None                   1.000000   
                      47    None                   1.000000   
             5        29    None                   0.664581   
                      47    None                   0.757572   

                            validation_log_likelihood_per_row  \
family       n_states seed                                      
normal_hmm   3        29                           -21.578788   
                      47                           -22.143909   
             4        29                           -19.738497   
                      47                           -19.738903   
             5        29                           -18.453852   
                      47                           -18.806643   
studentt_hmm 3        29                           -22.894412   
                      47                           -23.661815   
             4        29                           -21.071653   
                      47                           -21.071653   
             5        29                           -20.063935   
                      47                           -20.319191   

                            prefix_invariant  
family       n_states seed                    
normal_hmm   3        29                True  
                      47                True  
             4        29                True  
                      47                True  
             5        29                True  
                      47                True  
studentt_hmm 3        29                True  
                      47                True  
             4        29                True  
                      47                True  
             5        29                True  
                      47                True

Frozen external procedure: normal_hmm, K=4


## Apply the frozen procedure to every primary market

The primary comparison holds sampling at five minutes and the threshold grid at 0.20% × 1.5. Model parameters are refitted on each market's training partition, but selection and evaluation rules remain fixed. Every market receives the same validation occupancy, prefix, restart-stability, and variance diagnostics; ineligible markets stay in the table rather than disappearing.

In [6]:
model_records = []
primary_paths = {}
for symbol, market_class in MARKETS.items():
    symbol_prices = native_prices[symbol].loc[common_start:common_end]
    _, symbol_global, _ = build_multiscale_state(
        symbol_prices, PRIMARY_THRESHOLDS, include_scale_state=False
    )
    symbol_prepared = prepare_data(
        symbol_global, CLOCK, FEATURE_TIER, split=pilot_prepared.split,
        frozen_feature_columns=pilot_prepared.feature_columns,
    )
    external_selection = select_hmm_candidates(
        symbol_prepared,
        (SELECTED_FAMILY,),
        (SELECTED_N_STATES,),
        random_state=RANDOM_STATE,
        stability_seeds=STABILITY_SEEDS,
        max_iter=MAX_ITER,
        n_restarts=N_RESTARTS,
        minimum_validation_occupancy=MINIMUM_VALIDATION_OCCUPANCY,
        minimum_restart_ari=MINIMUM_RESTART_ARI,
        min_emission_variance=MIN_EMISSION_VARIANCE,
    )
    fit = external_selection.fits[0]
    record = external_selection.table.iloc[0].to_dict()
    record.update({
        "symbol": symbol,
        "market_class": market_class,
        "frozen_features_with_point_mass": tuple(
            column for column in pilot_prepared.feature_columns
            if symbol_prepared.training_point_mass_share[column] >= DEFAULT_MAX_POINT_MASS_SHARE
        ),
    })
    if fit.error is None:
        test_states = fit.states["test"]
        test_occupancy = np.bincount(test_states, minlength=SELECTED_N_STATES) / len(test_states)
        diagnostics = add_future_diagnostics(symbol_global, (24,))
        test_frame = symbol_prepared.frames["test"].copy()
        test_frame["model_state"] = test_states
        test_frame = test_frame.join(diagnostics[["future_realized_vol_24", "future_dcc_count_24"]])
        record.update({
            "test_log_likelihood_per_row": fit.log_likelihoods["test"] / len(test_states),
            "test_minimum_occupancy": test_occupancy.min(),
            "test_switch_rate": np.mean(test_states[1:] != test_states[:-1]),
            "test_mean_confidence": fit.probabilities["test"].max(axis=1).mean(),
        })
        partition_columns = {
            "model": "model_state",
            "structural": "structural_regime",
            "direction_aware": "directed_regime",
        }
        diagnostics_to_test = {
            "volatility": "future_realized_vol_24",
            "activity": "future_dcc_count_24",
        }
        for diagnostic_index, (diagnostic_name, value_column) in enumerate(diagnostics_to_test.items()):
            for partition_name, label_column in partition_columns.items():
                evidence = permutation_omega_test(
                    test_frame[value_column], test_frame[label_column], n_permutations=N_PERMUTATIONS,
                    random_state=RANDOM_STATE + diagnostic_index,
                )
                record[f"test_{diagnostic_name}_{partition_name}_omega_squared"] = evidence["omega_squared"]
                record[f"test_{diagnostic_name}_{partition_name}_permutation_p"] = evidence[
                    "permutation_p_value"
                ]
        incremental = incremental_usefulness(
            test_frame,
            list(diagnostics_to_test.values()),
            "model_state",
            {"structural": "structural_regime", "direction_aware": "directed_regime"},
            n_bootstrap=N_BOOTSTRAP,
            block_length=BOOTSTRAP_BLOCK_ROWS,
            confidence=BOOTSTRAP_CONFIDENCE,
            random_state=RANDOM_STATE,
        )
        for diagnostic_name, value_column in diagnostics_to_test.items():
            row = incremental.loc[value_column]
            record[f"test_{diagnostic_name}_best_baseline"] = row["best_baseline"]
            record[f"test_{diagnostic_name}_model_minus_best_baseline"] = row["model_minus_best_baseline"]
            record[f"test_{diagnostic_name}_ci_low"] = row["ci_low"]
            record[f"test_{diagnostic_name}_ci_high"] = row["ci_high"]
            record[f"test_{diagnostic_name}_verdict"] = row["verdict"]
        path_parts = []
        for split_name in ("train", "validation", "test"):
            part = symbol_prepared.frames[split_name].copy()
            part["model_state"] = fit.states[split_name]
            part["model_state_label"] = [f"state_{state}" for state in fit.states[split_name]]
            path_parts.append(part)
        primary_paths[symbol] = {
            "path": pd.concat(path_parts).sort_index(),
            "eligible": bool(record["eligible"]),
        }
    model_records.append(record)
    del symbol_global
    gc.collect()

model_robustness = pd.DataFrame.from_records(model_records).set_index("symbol")
eligibility_columns = [
    "minimum_state_occupancy", "minimum_restart_validation_ari",
    "filtering_prefix_invariant", "variance_floor_hit_share", "eligible", "baseline_nmi",
    "frozen_features_with_point_mass",
    "test_minimum_occupancy", "test_switch_rate", "test_mean_confidence",
]
display(model_robustness[eligibility_columns])
effect_columns = [
    column for column in model_robustness.columns
    if column.endswith("omega_squared") or column.endswith("permutation_p")
]
display(model_robustness[effect_columns])
incremental_columns = [
    column for column in model_robustness.columns
    if column.endswith(("_best_baseline", "_model_minus_best_baseline", "_ci_low", "_ci_high", "_verdict"))
]
print(
    "Model margin over the stronger direct baseline (block-bootstrap interval). "
    "'no_information_beyond_baseline' means the interval contains zero."
)
display(model_robustness[incremental_columns])

,minimum_state_occupancy,minimum_restart_validation_ari,filtering_prefix_invariant,variance_floor_hit_share,eligible,test_minimum_occupancy,test_switch_rate,test_mean_confidence
symbol,,,,,,,,
GBPUSD.r,0.156716,1.000000,True,0.059524,True,0.144536,0.880000,0.997735
EURUSD.r,0.199761,0.549906,True,0.071429,False,0.160982,0.804645,0.979922
USDJPY.r,0.166391,0.720680,True,0.059524,False,0.174393,0.882873,0.997751
US500.r,0.152425,0.589681,True,0.071429,False,0.164349,0.832382,0.982544
XAUUSD.r,0.190941,0.709503,True,0.059524,False,0.181405,0.902620,0.998030


,test_volatility_model_omega_squared,test_volatility_model_permutation_p,test_volatility_structural_omega_squared,test_volatility_structural_permutation_p,test_volatility_direction_aware_omega_squared,test_volatility_direction_aware_permutation_p,test_activity_model_omega_squared,test_activity_model_permutation_p,test_activity_structural_omega_squared,test_activity_structural_permutation_p,test_activity_direction_aware_omega_squared,test_activity_direction_aware_permutation_p
symbol,,,,,,,,,,,,
GBPUSD.r,0.008784,0.022,0.006527,0.042,0.007917,0.062,0.004075,0.116,0.016408,0.002,0.018416,0.002
EURUSD.r,0.020708,0.002,0.013196,0.008,0.009101,0.056,0.003578,0.120,0.001209,0.274,0.000000,1.000
USDJPY.r,0.036689,0.002,0.015649,0.002,0.114802,0.002,0.023228,0.002,0.007609,0.022,0.082739,0.002
US500.r,0.050400,0.002,0.000000,1.000,0.013144,0.002,0.046467,0.002,0.002933,0.008,0.010385,0.002
XAUUSD.r,0.013181,0.002,0.009022,0.002,0.011090,0.002,0.003000,0.002,0.000980,0.026,0.002325,0.004


## Recent primary-market casebooks

Visual inspection asks whether inferred states correspond to intelligible price/hierarchy episodes in every market. These are regime markers, not trades.

In [7]:
if PLOT_RECENT_PRIMARY_CASES:
    for symbol, case in primary_paths.items():
        eligibility = "eligible" if case["eligible"] else "ineligible"
        plot_price_states(
            case["path"],
            "model_state_label",
            f"{symbol}: frozen-procedure raw causal states ({eligibility} on validation)",
            last_n=LAST_N,
        ).show()
else:
    print("Recent cross-market casebook plots disabled by configuration.")

# Phase 24 completion gate

1. All 45 structural cases are retained, including inadequate cases.
2. Threshold ladders and frequencies are fixed across symbols rather than optimized per market.
3. Common calendar coverage is used.
4. Period instability is reported through occupancy ranges across three calendar thirds shared by every market and sampling frequency.
5. The HMM family and state count are selected on GBP validation only after prefix, occupancy, and restart gates.
6. External markets refit the frozen procedure but do not reselect it.
7. The same validation eligibility evidence is reported for every external market; failures remain visible.
8. Held-out diagnostic separation uses omega-squared and permutation evidence against structural and direction-aware baselines, plus a block-bootstrap interval for the model's margin over the stronger baseline; a market whose interval contains zero is reported as adding no information beyond the baseline.
9. Variance-floor hits, confidence, persistence, and occupancy are interpreted together rather than separately.
10. Price-path casebooks keep raw state IDs visible and identify ineligible cases.
11. A result requiring different rules or stories for every symbol is non-transfer, not robustness.
12. These previously inspected markets cannot be represented as untouched external confirmation.